# Disambiguation endpoint smoke test

This notebook runs the full flow using the API endpoints:
1) document extraction
2) NER prediction per paragraph
3) disambiguation


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import requests


In [ ]:
user_prompt_template= """
Se proporciona la lista de entidades (personas) pre-clusterizadas. Cada una incluye en `attributes['context']` las ventanas de texto de ±120 caracteres donde fue mencionada en la causa judicial para que puedas determinar su rol y validez.

# Entidades Pre-clusterizadas a validar:
{canonical_entities}

Procesa la lista siguiendo las instrucciones de filtrado, fusión y asignación de roles.
"""

In [ ]:
system_prompt = """
Eres un auditor experto en desambiguación de entidades legales. Tu tarea es validar y enriquecer una lista de personas pre-agrupadas.

### Instrucciones Estrictas:
1. **Asignación de Rol:** Identifica el rol procesal basándote en las ventanas de contexto dentro de cada entidad canónica en la parte de `attributes`['context']
  Elija un rol EXCLUSIVAMENTE en esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
2. **Regla "Dr/a":** Si se menciona como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito, asígnale el rol "Abogado/a".
3. **Validación y Fusión:** - Usa los fragmentos de contexto para confirmar que todos los aliases de un grupo pertenecen a la misma persona.
   - Si detectas que dos entidades distintas son en realidad la misma persona (ej: un grupo con el nombre completo y otro con iniciales o cargo), fusiónalos.
   - Si un alias dentro de un grupo pertenece a una persona diferente, sepáralo.
4. **Filtrado:** Elimina cualquier entidad que no sea una persona física (ej: instituciones, direcciones, leyes).
5. **Limpieza:** Limpiá los `aliases` y el `canonical_text` si los mismos tienen otras palabras, pero respeta que estén escritos de igual manera a que sus ventanas de contexto.
  Te dejo un ejemplo de esta instrucción.
    Vos recibís:
        {
        "canonical_text": "por DREXLER, JORGE",
        "aliases": [
            "por DREXLER, JORGE",
            "Jorge Drexler"
        ],
        "attributes": {
            "context": [
                "Fdo. por DREXLER, JORGE - JUEZ DE CÁMARA el mismo cita en el documento 56 que Juan es culpable",
                "es acaso Jorge Drexler el juez designado para esta causa"
            ]
        }
    
    Debés entregar:
        {
        "canonical_text": "DREXLER, JORGE",
        "aliases": [
            "DREXLER, JORGE",
            "Jorge Drexler"
        ],
        "attributes": {
            "role": "Juez/a"
            ]
        }
        
6. **Manejo de Iniciales:** Identifica si las siglas (ej: "M.L.") corresponden a una persona con nombre completo en el listado y únelas si el contexto lo confirma.

### Formato de Salida (JSON):
Devuelve un array de objetos con esta estructura:
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Nombre de la entidad,
    "aliases": [
      "Nombre de la entidad",
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol de la lista o null"
    }
  }
]

No incluyas el campo 'context' en tu respuesta final.
"""

In [ ]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
DATA_ROOT = Path(
    os.getenv(
        "DISAMBIGUATION_DATA_ROOT",
        "../../../resources/data/restricted/disambiguation-eval/files",
    )
)
DOC_EXTENSIONS = {".pdf", ".docx"}

LIMIT = int(os.getenv("DISAMBIGUATION_DOC_LIMIT", "0"))  # 0 = no limit
TARGET_LABELS = os.getenv("DISAMBIGUATION_TARGET_LABELS", "PER")
TARGET_LABELS = [label.strip() for label in TARGET_LABELS.split(",") if label.strip()]

FUZZY_THRESHOLD = int(os.getenv("DISAMBIGUATION_THRESHOLD", "70"))
FUZZY_SCORER = os.getenv("DISAMBIGUATION_SCORER", "token_set_ratio")
FUZZY_PROCESSOR = os.getenv("DISAMBIGUATION_PROCESSOR", "light_normalizer")
MODEL = os.getenv("MODEL", 'phi4:14b')
MODEL_CONTEXT = os.getenv("MODEL_CONTEXT", 9_500)
CONTEXT_WINDOW_LENGTH = os.getenv("CONTEXT_WINDOW_LENGTH", 120)
SYSTEM_PROMPT = os.getenv("SYSTEM_PROMPT", system_prompt)
USER_PROMPT_TEMPLATE = os.getenv("USER_PROMPT_TEMPLATE", user_prompt_template)
TOKEN_LIMIT_FRAC = os.getenv("TOKEN_LIMIT_FRAC", 2/3)
TOKENIZER_MODEL = os.getenv("TOKENIZER_MODEL", "microsoft/phi-4")

print(f"API: {API_BASE_URL}")
print(f"Data root: {DATA_ROOT}")
print(f"Target labels: {TARGET_LABELS or 'ALL'}")
print(
    f"Fuzzy params: scorer={FUZZY_SCORER}, threshold={FUZZY_THRESHOLD}, processor={FUZZY_PROCESSOR}"
)


In [ ]:
from aymurai.experiments.entity_disambiguation.runner import (
    call_extraction_api as extract_document,
)

def discover_documents(root: Path, extensions: set[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )

def predict_paragraphs(paragraphs: list[str]) -> list[dict]:
    endpoint = f"{API_BASE_URL}/anonymizer/predict"
    predictions = []
    for paragraph in paragraphs:
        response = requests.post(endpoint, json={"text": paragraph}, timeout=60)
        response.raise_for_status()
        predictions.append(response.json())
    return predictions

def disambiguate(predictions: list[dict]) -> list[dict]:
    endpoint = f"{API_BASE_URL}/anonymizer/disambiguate"
    params = {
        "threshold": FUZZY_THRESHOLD,
        "scorer": FUZZY_SCORER,
        "processor": FUZZY_PROCESSOR,
        "target_labels": TARGET_LABELS,
        "system_prompt": SYSTEM_PROMPT,
        "user_prompt_template": USER_PROMPT_TEMPLATE,
        "model": MODEL,
        "model_context": MODEL_CONTEXT,
        "context_window_length": CONTEXT_WINDOW_LENGTH,
        "token_limit_frac": TOKEN_LIMIT_FRAC,
        "tokenizer_model": TOKENIZER_MODEL
    }
    if TARGET_LABELS:
        params["target_labels"] = TARGET_LABELS
    response = requests.post(endpoint, params=params, json=predictions, timeout=600)
    response.raise_for_status()
    return response.json()


In [ ]:
documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
if LIMIT > 0:
    documents = documents[:LIMIT]

print(f"Found {len(documents)} documents")
documents[1:2]

In [ ]:
for doc_path in documents[1:2]:
    print(f"\n=== {doc_path.name} ===")
    session = requests.Session()
    document = extract_document(
        session,
        endpoint=f"{API_BASE_URL}/misc/document-extract",
        file_path=doc_path,
        timeout_s=300,
    )
    paragraphs = document["detail"]["document"]
    print(f"Paragraphs: {len(paragraphs)}")

    predictions = predict_paragraphs(paragraphs)
    print(f"Predictions: {len(predictions)}")

    disambiguated = disambiguate(predictions)
    print(json.dumps(disambiguated, indent=2, ensure_ascii=False))